# Panel setup

Prepare the regression panel from knowledge matrices already produced by
`know_calc_loop`. Run this notebook from top to bottom after selecting the run in
`project_settings.py`.

The three stages are:
1. Map the existing matrices and auxiliary files to the common sector classification.
2. Split incoming knowledge into four origin channels and build the knowledge panel.
3. Merge the knowledge panel with STAN data and add the variables used in estimation.

## Before running

- Set `BASE`, `YEARS`, `RD_UNIT`, `DIMENSION`, the technology selection, and the
  shock settings in `project_settings.py`.
- Direct and embodied channels are built together for the physical bases; running
  the full notebook twice just to switch between `dir` and `embod` is unnecessary.
- For CAPITAL, all three measures are included together: direct, embodied supply-chain,
  and embodied investment. Their source matrices must already exist for every selected year.
  `METRIC_WANTED` chooses the accessible-knowledge intensity and the downstream estimation;
  it does not filter which CAPITAL measures are loaded into this panel.
- For an FDI sector panel, use `BASE = "FDI"` and `METRIC_WANTED = "dir"`. Each selected year needs the inward matrix from `know_calc_loop`. Its columns identify the host country and sector receiving knowledge. Outward columns identify the investing country only. The reference estimator uses the inward total in specifications (1) and (5).
- Keep the STAN intermediate files and `intensite_propre.csv` in
  `code/panel_estimation/Panels_intermediaires/`. In `project_settings.py`, set
  `PANEL_CODE_DIR = PATH_WRK + "code/panel_estimation/"` after moving the code.
- Make the directory containing `project_settings.py` available to the notebook's
  Python kernel so that the import below succeeds.
- For a shock run, prepare the files selected by `AGG_SUFFIX` beforehand: the
  aggregation stage below writes baseline `_agg.csv` files, not shocked matrices.

## Outputs and next step

| Output | Purpose |
| --- | --- |
| `_agg.csv` files beside the source matrices and auxiliary files | Common sector mapping |
| `RESULTS_DIR/know_panels/panel_estimation_{BASE}.csv` | Knowledge channels and available trade/production variables |
| `RESULTS_DIR/regression_panels/panel_{BASE}_intermediaire_F.csv` | Intermediate panel after adding broad sectors C and H |
| `RESULTS_DIR/regression_panels/panel_{BASE}_F.csv` | Final panel, including own R&D intensity |

The notebooks in `code/panel_estimation/estimation_notebooks/` must read the final
file from `regression_panels/`. Re-running this notebook replaces the outputs for
the selected run and base. No regression is estimated here.

## Imports and project settings

Reload the central settings so that changes saved before execution are applied.
The printed paths identify the selected run and shock scenario.

In [9]:
import glob
from pathlib import Path
import os
import pandas as pd
import numpy as np
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, SRC_DIRS, SRC_DIR, METRIC_SPE, YEARS, RESULTS_DIR, RUN_DIR, CHOC, CHOC_SCENARIO, AGG_SUFFIX, METRIC_WANTED, PANEL_CODE_DIR
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO}): matrices *{AGG_SUFFIX}.csv" if CHOC else ": baseline matrices *_agg.csv"))
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO})" if CHOC else "") + f" -> {RESULTS_DIR}")

CHOC = False: baseline matrices *_agg.csv
CHOC = False -> D:/ENTICE/runs/SRDQ (c,s)/


# 1. Aggregate to common mapping

Read `mapping/final_aggregation/map_agg_{BASE}.xlsx` and map the source sectors to
the common classification. The functions below handle MARIO matrices, patent
citations, FDI, vectors, and auxiliary production/trade files according to their
layouts. Sectors missing from a mapping are reported and kept unchanged.

This stage processes the raw files found in `SRC_DIRS`, including available years
outside `YEARS`. Files already bearing an `_agg` suffix are skipped as inputs.

In [10]:
# folders written by know_calc_loop: "SRDV (c,s)" / "SRDQ (c,s)", or one folder per technology of TECHS in (c,s,t)
# (RD_UNIT, DIMENSION and TECHS are set in project_settings.py)
OUT_DIRS = SRC_DIRS
MAP_DIR = PATH_WRK + "mapping/final_aggregation"
BASES = [BASE]
# BASES = ["FIGARO", "EXIOBASE", "CAPITAL", "GTAP", "FDI", "OECD_ICIO", "Patents_Citation"]

MATRIX_PREFIXES = ["rd_dir_CI", "rd_embod_CI", "rd_imp_CI", "rd_tot_CI", "z_CI", "rd_dir_CAPITAL", "rd_embod_CAPITAL",
                   "rd_embod_investment_CAPITAL", "rd_dir_cit", "rd_dir_fdips", "rd_dir_fdips_inwards"]

# Patent and FDI matrices use the layouts below.
# - rd_dir_cit (Patents_Citation): only 2 column levels (Region, Sector) instead of 3 (Region, Level, Item),
#   and an extra "Year" level on the rows (Year, Region, Sector)
# - rd_dir_fdips (FDI): rows = host (Region, Sector), columns = investing country only (1 level, no sector)

# - rd_dir_fdips_inwards (FDI): both axes carry (Region, Sector).

# Wide files with Region[, Sector] identifiers and year columns, separated by ";"
# -> a single file (all years as columns), in SRDQ (c,s)/{base}/output_loop_{base}/
LONG_SECTOR_FILES = ["production_{base}.csv", "exports_{base}.csv", "imports_intermediate_{base}.csv"]

# Wide files with no Sector column (Region + year columns only) -> no sector
# aggregation needed, just copied as-is under the _agg name
NO_SECTOR_FILES = ["imports_final_demand_{base}.csv"]


def load_mapping(base):
    p = os.path.join(MAP_DIR, f"map_agg_{base}.xlsx")
    df = pd.read_excel(p, sheet_name="Sector", dtype=str)
    # column "Sector" = the base's own aggregation target (source, e.g. "C20_C21")
    # column "aggregation" = the common final sector (destination, e.g. "C20+C21")
    return dict(zip(df["Sector"], df["aggregation"]))


def warn_unmapped(items, mapping, source):
    # sectors present in the file but missing from the base's map_agg:
    # kept as-is (identity) instead of raising -- the state of the mapping
    # is left to the user to handle.
    missing = sorted(set(items) - set(mapping))
    if missing:
        print(f"  [warning] {os.path.basename(source)}: sectors missing from map_agg (kept as-is) -> {missing}")
    return missing


def apply_mapping(values, mapping):
    return [mapping.get(v, v) for v in values]


def aggregate_matrix_file(path, mapping):
    df = pd.read_csv(path, sep=";", header=[0, 1, 2], index_col=[0, 1, 2])
    df.index.names = df.columns.names = ["Region", "Level", "Item"]
    warn_unmapped(df.index.get_level_values("Item").unique(), mapping, path)

    idx = df.index.to_frame(index=False)
    idx["Item"] = apply_mapping(idx["Item"], mapping)
    df.index = pd.MultiIndex.from_frame(idx)

    cols = df.columns.to_frame(index=False)
    cols["Item"] = apply_mapping(cols["Item"], mapping)
    df.columns = pd.MultiIndex.from_frame(cols)

    df = df.groupby(level=["Region", "Level", "Item"]).sum()               # rows
    df = df.T.groupby(level=["Region", "Level", "Item"]).sum().T           # columns
    return df.sort_index().sort_index(axis=1)


def aggregate_citation_matrix_file(path, mapping):
    # columns: 2 levels (Region, Sector) ; rows: 3 levels (Year, Region, Sector)
    df = pd.read_csv(path, sep=";", header=[0, 1], index_col=[0, 1, 2])
    df.index.names = ["Year", "Region", "Sector"]
    df.columns.names = ["Region", "Sector"]
    warn_unmapped(df.index.get_level_values("Sector").unique(), mapping, path)

    idx = df.index.to_frame(index=False)
    idx["Sector"] = apply_mapping(idx["Sector"], mapping)
    df.index = pd.MultiIndex.from_frame(idx)

    cols = df.columns.to_frame(index=False)
    cols["Sector"] = apply_mapping(cols["Sector"], mapping)
    df.columns = pd.MultiIndex.from_frame(cols)

    df = df.groupby(level=["Year", "Region", "Sector"]).sum()               # rows
    df = df.T.groupby(level=["Region", "Sector"]).sum().T                   # columns
    return df.sort_index().sort_index(axis=1)


def aggregate_fdi_matrix_file(path, mapping):
    # rows: 2 levels (Region, Sector) = host region and sector ; columns: 1 level = investing country (no sector level:
    # only the sectors of the rows are aggregated)
    df = pd.read_csv(path, sep=";", index_col=[0, 1])
    df.index.names = ["Region", "Sector"]
    warn_unmapped(df.index.get_level_values("Sector").unique(), mapping, path)

    idx = df.index.to_frame(index=False)
    idx["Sector"] = apply_mapping(idx["Sector"], mapping)
    df.index = pd.MultiIndex.from_frame(idx)

    df = df.groupby(level=["Region", "Sector"]).sum()                       # rows
    return df.sort_index().sort_index(axis=1)


def read_fdi_inward_matrix(path):
    matrix = pd.read_csv(path, sep=";", header=[0, 1], index_col=[0, 1])
    axes = ["Region", "Sector"]
    if matrix.index.names != axes or matrix.columns.names != axes:
        raise ValueError(f"Expected (Region, Sector) on both axes of the inward FDI matrix: {path}")
    return matrix


def aggregate_fdi_inward_matrix_file(path, mapping):
    matrix = read_fdi_inward_matrix(path)
    source_sectors = matrix.index.get_level_values("Sector").unique()
    dest_sectors = matrix.columns.get_level_values("Sector").unique()
    warn_unmapped(source_sectors.union(dest_sectors), mapping, path)
    matrix = matrix.rename(index=mapping, columns=mapping, level="Sector")
    matrix = matrix.groupby(level=["Region", "Sector"]).sum()
    return matrix.T.groupby(level=["Region", "Sector"]).sum().T


AGGREGATORS = {
    "rd_dir_cit": aggregate_citation_matrix_file,
    "rd_dir_fdips": aggregate_fdi_matrix_file,
    "rd_dir_fdips_inwards": aggregate_fdi_inward_matrix_file,
}


def aggregate_vector_file(path, mapping):
    # vector_{metric}_{base}_{year}.csv (OUTPUT_TYPE "vector" of the main program): one row per destination, last column = value.
    # Keys (Region, Level, Item) for the mario-based bases, (Region, Sector) for patents and inward FDI, Region only for outward FDI
    # (no sector: copied as-is)
    df = pd.read_csv(path, sep=";", dtype={"Region": str, "Level": str, "Item": str, "Sector": str})
    keys, value = list(df.columns[:-1]), df.columns[-1]
    sector = next((k for k in ("Item", "Sector") if k in keys), None)
    if sector is None:
        return df
    warn_unmapped(df[sector].unique(), mapping, path)
    df[sector] = apply_mapping(df[sector], mapping)
    return df.groupby(keys, as_index=False)[value].sum().sort_values(keys)


def aggregate_long_file(path, mapping):
    df = pd.read_csv(path, sep=";", dtype={"Region": str, "Sector": str})
    warn_unmapped(df["Sector"].unique(), mapping, path)
    df["Sector"] = apply_mapping(df["Sector"], mapping)
    year_cols = [c for c in df.columns if c not in ("Region", "Sector")]
    return df.groupby(["Region", "Sector"], as_index=False)[year_cols].sum()


def out_path(path):
    root, ext = os.path.splitext(path)
    return root + "_agg" + ext


def is_already_aggregated(path):
    return "_agg" in os.path.basename(path)   # _agg.csv and _agg_choc.csv are results of an aggregation


def run_base(base, out_dir):
    mapping = load_mapping(base)
    done = 0
    base_dir = os.path.join(out_dir, base)
    if not os.path.isdir(base_dir):
        print(f"[{base}] folder missing, skipped: {base_dir}")
        return

    for year_dir in sorted(glob.glob(os.path.join(base_dir, f"output_loop_{base}_*"))):
        for prefix in MATRIX_PREFIXES:
            aggregator = AGGREGATORS.get(prefix, aggregate_matrix_file)
            for f in glob.glob(os.path.join(year_dir, f"{prefix}_{base}_*.csv")):
                if is_already_aggregated(f):
                    continue
                aggregator(f, mapping).to_csv(out_path(f), sep=";")
                done += 1
            for f in glob.glob(os.path.join(year_dir, f"vector_{prefix}_{base}_*.csv")):
                if is_already_aggregated(f):
                    continue
                aggregate_vector_file(f, mapping).to_csv(out_path(f), sep=";", index=False)
                done += 1

    no_year_dir = os.path.join(base_dir, f"output_loop_{base}")
    if os.path.isdir(no_year_dir):
        for pattern in LONG_SECTOR_FILES:
            f = os.path.join(no_year_dir, pattern.format(base=base))
            if os.path.exists(f) and not is_already_aggregated(f):
                aggregate_long_file(f, mapping).to_csv(out_path(f), sep=";", index=False)
                done += 1

        for pattern in NO_SECTOR_FILES:
            f = os.path.join(no_year_dir, pattern.format(base=base))
            if os.path.exists(f) and not is_already_aggregated(f):
                pd.read_csv(f, sep=";").to_csv(out_path(f), sep=";", index=False)
                done += 1

    print(f"[{base}] _agg files created: {done} ({out_dir})")


### Run aggregation

Execute the aggregation functions for each configured source directory and base.


In [11]:
for out_dir in OUT_DIRS:
    for b in BASES:
        try:
            run_base(b, out_dir)
        except Exception as e:
            if b == "FDI":
                raise  # Stop before reading an older FDI aggregate after a failed rebuild.
            print(f"[{b}] ERROR ({out_dir}): {e}")

[FDI] _agg files created: 112 (D:/ENTICE/SRDQ (c,s)/)


# 2. Compute knowledge blocks

Read the mapped knowledge matrices for the selected run, base, and years. Matrix
rows identify the origin of knowledge; columns identify the receiving activity.
The resulting panel has one observation per destination region, sector, and year.

### Load a knowledge matrix

Use the matrix layout appropriate to the base and remove extra index levels so
that both axes are labelled `(Region, Sector)`. `AGG_SUFFIX` selects the baseline
or already-prepared shock matrices. FDI sector panels use the inward matrix. Its rows describe the source of knowledge and its columns identify the recipient country and sector.

In [12]:
def load_your_matrix(metric: str, year: int, base: str) -> pd.DataFrame:
    """
    Load a knowledge matrix exported by know_calc_loop and mapped to common sectors.
    Remove the MARIO Level or patent Year level, leaving Region and Sector on both axes.
    """

    phys = ["FIGARO", "EXIOBASE", "GTAP", "OECD_ICIO", "CAPITAL"]

    file_path = Path(SRC_DIR) / base / f"output_loop_{base}_{year}" / f"{metric}_{base}_{year}{AGG_SUFFIX}.csv"   # SRC_DIR: folder of the run chosen in project_settings; AGG_SUFFIX: _agg (baseline) or _agg_choc (CHOC = True)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    if base == "FDI":
        return read_fdi_inward_matrix(file_path)
    if base in phys:
        matrix = pd.read_csv(
            file_path,
            sep=";",
            header=[0, 1, 2],
            index_col=[0, 1, 2],
        )

        matrix.index = matrix.index.droplevel("Level")
        matrix.columns = matrix.columns.droplevel("Level")
    else:
        matrix = pd.read_csv(
            file_path,
            sep=";",
            header=[0, 1],
            index_col=[0, 1, 2],
        )

        if "Year" in matrix.index.names or matrix.index.nlevels == 3:
            matrix.index = matrix.index.droplevel(0)

    matrix.index.names = ["Region", "Sector"]
    matrix.columns.names = ["Region", "Sector"]

    return matrix

### Define the four origin channels

For each destination, sum incoming knowledge over four mutually exclusive groups:
same sector/same country, same sector/other country, other sector/same country,
and other sector/other country. Their sum equals the destination's total incoming
knowledge.

In [13]:
def compute_4_blocks(matrix: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    Matrix rows identify origins (Region, Sector); columns identify destinations.

    Return one row per destination (Region, Sector), with four columns that
    partition all incoming knowledge by origin country and sector.
    """
    regions_rows = matrix.index.get_level_values("Region").to_numpy()
    sectors_rows = matrix.index.get_level_values("Sector").to_numpy()
    regions_cols = matrix.columns.get_level_values("Region").to_numpy()
    sectors_cols = matrix.columns.get_level_values("Sector").to_numpy()

    same_region = regions_rows[:, None] == regions_cols[None, :]
    same_sector = sectors_rows[:, None] == sectors_cols[None, :]

    values = matrix.to_numpy()

    def block_sum(mask):
        return pd.Series(np.where(mask, values, 0.0).sum(axis=0), index=matrix.columns)

    out = pd.DataFrame({
        f"rd_same_sector_same_country_{metric}": block_sum(same_sector & same_region),
        f"rd_same_sector_diff_country_{metric}": block_sum(same_sector & ~same_region),
        f"rd_diff_sector_same_country_{metric}": block_sum(~same_sector & same_region),
        f"rd_diff_sector_diff_country_{metric}": block_sum(~same_sector & ~same_region),
    })
    out.index.names = ["Region", "Sector"]
    return out

### Define the trade and production merge

Read the mapped auxiliary files, convert year columns to long format, and join
exports, imports, and production to each destination observation. These variables
are added for the physical bases.

In [14]:
def add_trade_and_production(path_wrk: str, year: list, base: str, df: pd.DataFrame) -> pd.DataFrame:
    """Merge mapped trade and production series into the knowledge panel."""
    def load_and_melt(name, value_name, id_vars):
        path = SRC_DIR + f"{base}/output_loop_{base}/{name}_{base}_agg.csv"
        wide = pd.read_csv(path, header=0, sep=";")
        year_cols = [c for c in wide.columns if c not in id_vars]
        long = wide.melt(
            id_vars=id_vars,
            value_vars=year_cols,
            var_name="year",
            value_name=value_name,
        )
        long["year"] = long["year"].astype(int)
        return long

    exports = load_and_melt("exports", "exports", ["Region", "Sector"])
    imports_final_demand = load_and_melt("imports_final_demand", "imports_final_demand", ["Region"])
    imports_intermediate = load_and_melt("imports_intermediate", "imports_intermediate", ["Region", "Sector"])
    production = load_and_melt("production", "production", ["Region", "Sector"])

    df_complete = df.reset_index()

    for y in year:
        if y not in exports["year"].values:
            raise AssertionError(f"Year {y} not in {base}_exports")
        if y not in imports_final_demand["year"].values:
            raise AssertionError(f"Year {y} not in {base}_imports_final_demand")
        if y not in imports_intermediate["year"].values:
            raise AssertionError(f"Year {y} not in {base}_imports_intermediate")
        if y not in production["year"].values:
            raise AssertionError(f"Year {y} not in {base}_production")

    df_complete = df_complete.merge(exports, on=["Region", "Sector", "year"], how="left")
    df_complete = df_complete.merge(imports_final_demand, on=["Region", "year"], how="left")
    df_complete = df_complete.merge(imports_intermediate, on=["Region", "Sector", "year"], how="left")
    df_complete = df_complete.merge(production, on=["Region", "Sector", "year"], how="left")

    df_complete = df_complete[df_complete["year"].isin(year)]
    df_complete = df_complete.set_index(["Region", "Sector", "year"])

    return df_complete

### Build and save the knowledge panel

Compute the channels for each year, combine direct and embodied measures where
available, and always add CAPITAL investment channels for that database. Save the result in
`know_panels/`; the next stage reads this file.

In [15]:
METRIC = METRIC_SPE[f"{BASE}_dir"]
panel_rows = []
for year in YEARS:
    matrix = load_your_matrix(METRIC, year, BASE)
    df_year = compute_4_blocks(matrix, METRIC)
    df_year["year"] = year
    panel_rows.append(df_year.reset_index())
panel = pd.concat(panel_rows, ignore_index=True).set_index(["Region", "Sector", "year"])

if BASE not in ["Patents_Citation", "FDI"]:
    METRIC = METRIC_SPE[f"{BASE}_embod"]
    panel_rows2 = []
    for year in YEARS:
        matrix2 = load_your_matrix(METRIC, year, BASE)
        df_year2 = compute_4_blocks(matrix2, METRIC)
        df_year2["year"] = year
        panel_rows2.append(df_year2.reset_index())
    panel2 = pd.concat(panel_rows2, ignore_index=True).set_index(["Region", "Sector", "year"])

    panel = pd.concat([panel, panel2], axis=1)

    if BASE == "CAPITAL":
        metric_investment = METRIC_SPE["CAPITAL_embod_investment"]
        panel_rows_investment = []

        for year in YEARS:
            matrix_investment = load_your_matrix(
                metric_investment, year, BASE
            )
            df_investment = compute_4_blocks(
                matrix_investment, metric_investment
            )
            df_investment["year"] = year
            panel_rows_investment.append(df_investment.reset_index())

        panel_investment = pd.concat(
            panel_rows_investment, ignore_index=True
        ).set_index(["Region", "Sector", "year"])

        panel = pd.concat([panel, panel_investment], axis=1)

    panel = add_trade_and_production(PATH_WRK, YEARS, BASE, panel)

os.makedirs(RESULTS_DIR + "know_panels", exist_ok=True)
panel.to_csv(RESULTS_DIR + f"know_panels/panel_estimation_{BASE}.csv")
print(f"run {RUN_DIR} | {BASE} | years {YEARS} -> {RESULTS_DIR}know_panels/panel_estimation_{BASE}.csv")

run SRDQ (c,s) | FDI | years [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022] -> D:/ENTICE/runs/SRDQ (c,s)/know_panels/panel_estimation_FDI.csv


# 3. Build the final regression panel

Add broad sectors C and H, merge with the STAN cost and factor-price data, and
construct the knowledge aggregates, the 2009 indicator, production concentration,
and R&D intensity measures. The final export includes all these variables and
does not split countries into low- and high-intensity groups.

### Read and validate the knowledge panel

Reload the file just produced. If CAPITAL investment is selected, check that all
four investment channels are present before continuing.

In [16]:
panel_file = RESULTS_DIR + f"know_panels/panel_estimation_{BASE}.csv"
print("Reading knowledge panel:", panel_file)
df = pd.read_csv(panel_file)

# The investment channels must already be present in the knowledge panel created above.
if BASE == "CAPITAL":
    investment_metric = METRIC_SPE["CAPITAL_embod_investment"]
    investment_columns = [
        f"rd_same_sector_same_country_{investment_metric}",
        f"rd_same_sector_diff_country_{investment_metric}",
        f"rd_diff_sector_same_country_{investment_metric}",
        f"rd_diff_sector_diff_country_{investment_metric}",
    ]
    missing = [c for c in investment_columns if c not in df.columns]
    if missing:
        raise ValueError(
            f"Investment columns missing from the knowledge panel: {missing}. "
            "Re-run stage 2 with BASE='CAPITAL', using the same RESULTS_DIR. "
            f"Expected file: {panel_file}"
        )

Reading knowledge panel: D:/ENTICE/runs/SRDQ (c,s)/know_panels/panel_estimation_FDI.csv


In [17]:
print(df.columns)

Index(['Region', 'Sector', 'year',
       'rd_same_sector_same_country_rd_dir_fdips_inwards',
       'rd_same_sector_diff_country_rd_dir_fdips_inwards',
       'rd_diff_sector_same_country_rd_dir_fdips_inwards',
       'rd_diff_sector_diff_country_rd_dir_fdips_inwards'],
      dtype='str')


In [18]:
df = df.rename(columns={"year": "annee"})

### Add broad sectors C and H

Sum the selected additive columns by region and year across C subsectors to build
manufacturing (C). For physical bases, apply the same procedure to the configured
H subsectors. Keep the detailed sectors alongside these broad aggregates.

In [19]:
c_subsectors = df.loc[df['Sector'].str.startswith('C', na=False), 'Sector'].unique().tolist()

mask_c = df['Sector'].isin(c_subsectors)

phys = ["FIGARO", "EXIOBASE", "GTAP", "OECD_ICIO", "CAPITAL"]

if BASE in phys:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',

            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',

            'exports',
            'imports_final_demand',
            'imports_intermediate',
            'production']
else:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}']

if BASE == "CAPITAL" and "rd_same_sector_same_country_rd_embod_investment_CAPITAL" in df.columns:
    col += ["rd_same_sector_same_country_rd_embod_investment_CAPITAL",
            "rd_same_sector_diff_country_rd_embod_investment_CAPITAL",
            "rd_diff_sector_same_country_rd_embod_investment_CAPITAL",
            "rd_diff_sector_diff_country_rd_embod_investment_CAPITAL",]

c_agg = (
    df[mask_c]
    .groupby(['Region', 'annee'], as_index=False)
    [col]
    .sum()
)
c_agg['Sector'] = 'C'

df_final = pd.concat([df, c_agg], ignore_index=True)

In [20]:
if BASE in phys:
    h_subsectors = df.loc[df['Sector'].str.startswith('H', na=False), 'Sector'].unique().tolist()
    mask_h = df_final['Sector'].isin(h_subsectors)

    h_agg = (
        df_final[mask_h]
        .groupby(['Region', 'annee'], as_index=False)
        [col]
        .sum()
    )
    h_agg['Sector'] = 'H'

    df_final1 = pd.concat([df_final, h_agg], ignore_index=True)
else:
    df_final1 = df_final

### Save the intermediate sector panel

Store the panel with broad sectors before the STAN merge. This file is an
intermediate output; estimations use the final `panel_{BASE}_F.csv` file.

In [21]:
os.makedirs(RESULTS_DIR + 'regression_panels', exist_ok=True)
df_final1.to_csv(RESULTS_DIR + f'regression_panels/panel_{BASE}_intermediaire_F.csv')

In [22]:
df_final1

,Region,Sector,annee,rd_same_sector_same_country_rd_dir_fdips_inwards,rd_same_sector_diff_country_rd_dir_fdips_inwards,rd_diff_sector_same_country_rd_dir_fdips_inwards,rd_diff_sector_diff_country_rd_dir_fdips_inwards
0,AT,A,1995,0.0,24.334707,0.0,0.0
1,AT,B,1995,0.0,807.735995,0.0,0.0
2,AT,C10_C12,1995,0.0,419.338956,0.0,0.0
3,AT,C13_C15,1995,0.0,53.935792,0.0,0.0
4,AT,C16_C18,1995,0.0,99.172188,0.0,0.0
...,...,...,...,...,...,...,...
41407,US,C,2018,0.0,673976.970778,0.0,0.0
41408,US,C,2019,0.0,698027.307826,0.0,0.0
41409,US,C,2020,0.0,717077.336624,0.0,0.0
41410,US,C,2021,0.0,740237.473723,0.0,0.0


### Merge with STAN data

Read `panel_va_STAN.csv` for physical bases or `panel_cit1.csv` for the other bases,
then merge by region, sector, and year. STAN observations form the left side of
the merge. For physical bases, keep the source database's production separately
as `production_{BASE}`.

In [23]:
if BASE in phys:
    va_STAN = pd.read_csv(PATH_WRK + 'code/panel_estimation/Panels_intermediaires/panel_va_STAN.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]

    df_final1 = df_final1.rename(columns={"production": f"production_{BASE}"})
    df = va_STAN.merge(df_final1, how='left', on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"])
else:
    va_STAN = pd.read_csv(PATH_WRK + 'code/panel_estimation/Panels_intermediaires/panel_cit1.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]
    join = "inner" if BASE == "FDI" else "left"
    df = va_STAN.merge(df_final1, how=join, on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"], errors="ignore")
    if BASE == "FDI" and df.empty:
        raise ValueError("No FDI observations match STAN on Region, Sector and annee.")

df

,Region,Sector,annee,VA_V,VA_Q,VA_costFactors,comp_emp,emp,net_stock_capital,production,...,w_l,c_u,produit,TFP_2,Y_L,K_L,rd_same_sector_same_country_rd_dir_fdips_inwards,rd_same_sector_diff_country_rd_dir_fdips_inwards,rd_diff_sector_same_country_rd_dir_fdips_inwards,rd_diff_sector_diff_country_rd_dir_fdips_inwards
0,AT,A,1995,3843.823732,3410.753099,5023.693732,348.500000,38.626784,3.428939e+04,7664.729833,...,9.022237,1.126972,18529.125933,0.184075,88.300209,887.710127,0.0,24.334707,0.0,0.0
1,AT,A,1996,3588.173927,3354.544562,4309.291927,354.200000,38.047075,3.438198e+04,7436.299892,...,9.309520,1.069646,17560.377520,0.191029,88.168264,903.669525,0.0,24.756775,0.0,0.0
2,AT,A,1997,3693.656888,3478.446066,4251.222888,364.700000,38.474740,3.479585e+04,7714.325937,...,9.478946,1.061870,17767.512831,0.195776,90.408566,904.381705,0.0,25.342561,0.0,0.0
3,AT,A,1998,3590.057497,3625.242586,4249.225497,366.000000,37.816498,3.498068e+04,7467.688729,...,9.678316,0.990294,17435.518421,0.207923,95.864048,925.011188,0.0,25.142488,0.0,0.0
4,AT,A,1999,3616.638040,3843.736884,4315.897040,382.300000,39.327879,3.516822e+04,7480.581422,...,9.720840,0.940917,17146.074094,0.224176,97.735677,894.231267,0.0,24.548471,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13430,US,R+S+T,2015,403824.172649,433197.497956,393560.340571,267416.971634,10318.000000,1.022369e+06,670507.180879,...,25.917520,0.932194,48733.288089,8.889150,41.984638,99.085998,0.0,453.066946,0.0,0.0
13431,US,R+S+T,2016,420530.767816,438314.944498,409734.759237,278968.564174,10471.000000,1.047762e+06,701452.164244,...,26.642017,0.959426,49355.487347,8.880774,41.859893,100.063201,0.0,497.101703,0.0,0.0
13432,US,R+S+T,2017,435249.731220,449286.366872,423538.087025,289233.014358,10519.000000,1.062726e+06,727146.899842,...,27.496246,0.968758,49479.158469,9.080315,42.711890,101.029181,0.0,537.779262,0.0,0.0
13433,US,R+S+T,2018,457138.704255,465631.493211,444723.251620,304570.694274,10767.000000,1.082054e+06,763918.279636,...,28.287424,0.981761,50153.909337,9.284052,43.246168,100.497266,0.0,559.232013,0.0,0.0


### Construct knowledge aggregates and controls

For each available metric, combine the four channels into domestic (D), foreign
(F), same-sector (I), other-sector (X), and total knowledge. Add the 2009 indicator
and an HHI based on countries' production shares within each sector and year.
For CAPITAL, `embod` denotes supply-chain knowledge and `embod_investment` denotes
investment knowledge.

In [24]:
# Embodied knowledge; for CAPITAL, this is the supply-chain measure.
if BASE in phys:
    col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_F'] = df[col_F].sum(axis=1)
    df['rd_embod_D'] = df[col_D].sum(axis=1)

    col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_I'] = df[col_I].sum(axis=1)
    df['rd_embod_X'] = df[col_X].sum(axis=1)

    col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod'] = df[col_S].sum(axis=1)

# Direct knowledge.
col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_F'] = df[col_F].sum(axis=1)
df['rd_dir_D'] = df[col_D].sum(axis=1)

col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_I'] = df[col_I].sum(axis=1)
df['rd_dir_X'] = df[col_X].sum(axis=1)

col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir'] = df[col_S].sum(axis=1)

df["2009_crisis"]=(df["annee"]==2009).astype(int)

# col = 'production_EXIOBASE_mapF'
# df['share'] = df['production'] / df.groupby(['Sector', 'annee'])[col].transform('sum')
df['share'] = df['production'] / df.groupby(['Sector', 'annee'])['production'].transform('sum')
df['HHI'] = df.groupby(['Sector', 'annee'])['share'].transform(lambda s: (s**2).sum())

if BASE == "CAPITAL" and "rd_same_sector_same_country_rd_embod_investment_CAPITAL" in df.columns:
    same_same = "rd_same_sector_same_country_rd_embod_investment_CAPITAL"
    same_diff = "rd_same_sector_diff_country_rd_embod_investment_CAPITAL"
    diff_same = "rd_diff_sector_same_country_rd_embod_investment_CAPITAL"
    diff_diff = "rd_diff_sector_diff_country_rd_embod_investment_CAPITAL"

    df["rd_embod_investment_D"] = df[[same_same, diff_same]].sum(axis=1)
    df["rd_embod_investment_F"] = df[[same_diff, diff_diff]].sum(axis=1)
    df["rd_embod_investment_I"] = df[[same_same, same_diff]].sum(axis=1)
    df["rd_embod_investment_X"] = df[[diff_same, diff_diff]].sum(axis=1)
    df["rd_embod_investment"] = df[[same_same, same_diff, diff_same, diff_diff]].sum(axis=1)

In [25]:
df = df.drop(columns=['share'])

### Compute accessible-knowledge intensity

Divide the selected accessible-knowledge measure by STAN production. Keep the
existing column name `intensité` for compatibility with the estimation code.
This measure differs from the own R&D expenditure intensity added below.

In [26]:
df['intensité'] = df[f'rd_{METRIC_WANTED}'] / df[f'production']

### Compute own R&D expenditure intensity

Read `intensite_propre.csv`, prepared upstream from own R&D expenditure and STAN
production in current million euros, independently of the knowledge database and
technology. Compute `int_propre = 100 * own R&D expenditure / production` for each
region, sector, and year: a value of 2 means 2%. This variable is used in the
interaction in specification (5). Production values at or below 1 million euros
are excluded from this ratio, following the existing rule.

In [27]:
# Own R&D / production: RDV and PRODV are prepared once by prépa_va_STAN, independently of the base and technology.
# rd_propre is annual R&D expenditure in current EUR million; production is STAN production in current EUR million (annual exchange rates), including C and H.
df_int = (pd.read_csv(PANEL_CODE_DIR + "Panels_intermediaires/intensite_propre.csv")
            .rename(columns={"rd_propre_V": "rd_propre", "production_V": "production"}))

# Own R&D intensity = the sector's R&D expenditure / the sector's production.
df_int["intensité_vraie"] = df_int["rd_propre"] / df_int["production"]

# Annual own R&D intensity R/Y, expressed in percent, for the regression interaction.
df_int.loc[df_int["production"] <= 1.0, "intensité_vraie"] = np.nan   # Exclude near-zero production to avoid extreme intensity ratios.
int_annual = (df_int[["Region", "Sector", "annee", "intensité_vraie"]]
              .rename(columns={"intensité_vraie": "int_propre"}))
int_annual["int_propre"] = 100 * int_annual["int_propre"]

### Merge own intensity and export the final panel

Join `int_propre` by region, sector, and year, then write the complete regression
panel once to `regression_panels/panel_{BASE}_F.csv`. The estimation notebooks can
reuse this file for the reference, fine-tuned, and scaled approaches.

In [28]:
# Complete regression panel, including own R&D intensity.
df = df.merge(int_annual, on=["Region", "Sector", "annee"], how="left")

os.makedirs(RESULTS_DIR + "regression_panels", exist_ok=True)

df.to_csv(RESULTS_DIR + f"regression_panels/panel_{BASE}_F.csv", index=False)

## FDI outward exposure by investing country and year

For FDI runs, also build the input for specifications (1) and (5) using the national outward vector. Every sector in an investing country observes the same national exposure in a given year. Set `FDI_OUTWARD_YEARS = [2017]` in `project_settings.py` for a cross-section. Select several years only to study changes in R&D exposure using the fixed 2017 portfolio. If the inward panel already exists, run **Imports and project settings**, **Compute own R&D expenditure intensity**, and this cell.

In [29]:
if BASE == "FDI":
    from project_settings import FDI_OUTWARD_YEARS

    outward_years = sorted(set(FDI_OUTWARD_YEARS))
    if not outward_years:
        raise ValueError("Select at least one year in FDI_OUTWARD_YEARS.")
    yearly = []
    for year in outward_years:
        folder = Path(SRC_DIR) / "FDI" / f"output_loop_FDI_{year}"
        vector_file = folder / f"vector_rd_dir_fdips_FDI_{year}.csv"
        if vector_file.exists():
            # Some exports leave the investing-country column header blank.
            vec = pd.read_csv(vector_file, sep=";", index_col=0)["rd_dir_fdips"]
        else:
            matrix_file = folder / f"rd_dir_fdips_FDI_{year}.csv"
            matrix = pd.read_csv(matrix_file, sep=";", index_col=["Region", "Sector"])
            vec = matrix.sum(axis=0, min_count=1)
        yearly.append(vec.rename("rd_outwards").rename_axis("Region").reset_index().assign(annee=year))

    exposure = pd.concat(yearly, ignore_index=True)
    keys = ["Region", "Sector", "annee"]
    stan = pd.read_csv(Path(PANEL_CODE_DIR) / "Panels_intermediaires" / "panel_cit1.csv",
                       usecols=keys + ["c_u", "w_l", "C_k"])
    panel_out = stan.merge(exposure, on=["Region", "annee"], how="inner", validate="many_to_one")
    panel_out = panel_out.merge(int_annual, on=keys, how="left", validate="one_to_one")
    missing_years = set(outward_years).difference(panel_out["annee"])
    if missing_years:
        raise ValueError(f"No outward/STAN overlap for years {sorted(missing_years)}.")
    panel_out["2009_crisis"] = (panel_out["annee"] == 2009).astype(int)
    out_file = Path(RESULTS_DIR) / "regression_panels" / "panel_FDI_outwards_F.csv"
    out_file.parent.mkdir(parents=True, exist_ok=True)
    panel_out.to_csv(out_file, index=False)
    print(f"{out_file}: {len(panel_out)} rows, {panel_out.Region.nunique()} investing countries, years {outward_years}")

D:\ENTICE\runs\SRDQ (c,s)\regression_panels\panel_FDI_outwards_F.csv: 571 rows, 23 investing countries, years [2017]
